In [883]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error

In [884]:
clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")

display(clf_df.head())
display(reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [885]:
print("Classification dataset shape:", clf_df.shape)
print("Regression dataset shape:", reg_df.shape)

print("\nClassification dataset information:")
clf_df.info()

print("\nRegression dataset information:")
reg_df.info()

Classification dataset shape: (900, 7)
Regression dataset shape: (800, 7)

Classification dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               900 non-null    str    
 1   complexity_score        878 non-null    float64
 2   team_experience_years   864 non-null    float64
 3   priority                887 non-null    str    
 4   channel                 891 non-null    str    
 5   customer_tenure_months  869 non-null    float64
 6   escalated               900 non-null    int64  
dtypes: float64(3), int64(1), str(3)
memory usage: 49.3 KB

Regression dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id              

In [886]:
print("Classification missing values:")
display(clf_df.isnull().sum())

print("\nRegression missing values:")
display(reg_df.isnull().sum())

Classification missing values:


ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64


Regression missing values:


ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64

In [887]:
print("Classification target distribution:")
display(clf_df["escalated"].value_counts())

print("\nClassification target percentages:")
display(clf_df["escalated"].value_counts(normalize=True) * 100)

Classification target distribution:


escalated
0    709
1    191
Name: count, dtype: int64


Classification target percentages:


escalated
0    78.777778
1    21.222222
Name: proportion, dtype: float64

In [888]:
# experiment A

X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

print("Features:")
display(X_clf.head())

print("Target:")
display(y_clf.head())

Features:


,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,2.0,15.0,Medium,Email,61.818896
1,9.0,6.0,Critical,Chat,110.024317
2,10.0,8.0,High,Phone,36.311244
3,2.0,1.0,Medium,Phone,85.127855
4,8.0,14.0,Medium,Chat,18.225059


Target:


0    0
1    0
2    0
3    0
4    0
Name: escalated, dtype: int64

In [889]:
# Split the data into training and testing sets
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    stratify=y_clf,
    random_state=42
)

print("Training set size:", X_train_clf.shape)
print("Testing set size:", X_test_clf.shape)

Training set size: (720, 5)
Testing set size: (180, 5)


In [890]:
# Identify numerical and categorical columns(preprocess)
numeric_features_clf = X_train_clf.select_dtypes(include=["number"]).columns.tolist()

categorical_features_clf = X_train_clf.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", numeric_features_clf)
print("Categorical features:", categorical_features_clf)

Numerical features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


C:\Users\hp\AppData\Local\Temp\ipykernel_13124\2818486877.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_clf = X_train_clf.select_dtypes(include=["object", "category"]).columns.tolist()


In [891]:
# Build the preprocessing pipelines

numeric_transformer_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [892]:
categorical_transformer_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [893]:
# tell to use applies the numerical pipeline to numerical columns.
# tell to use the categorical pipeline to categorical columns.
preprocessor_clf = ColumnTransformer(transformers=[
    ("num", numeric_transformer_clf, numeric_features_clf),
    ("cat", categorical_transformer_clf, categorical_features_clf)
])

if we preprocess the entire dataset before splitting or before cross-validation, information from the validation or test data could influence the preprocessing process. This is called data leakage.

For example, if we calculate the mean or median using all the data, the preprocessing step has already seen information from data that should have been kept separate.

By placing preprocessing inside a pipeline, the preprocessing steps are fitted using only the training portion of each CV fold, and then applied to the validation portion.

This helps prevent data leakage and gives us a more reliable evaluation.

now

Build and compare classification models

we will compare three machine learning models to see which one performs better at predicting whether a ticket will be escalated.

In [894]:
# Create the 5-fold Stratified Cross-Validation strategy

cv_clf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [895]:
# Build the Logistic Regression model

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", LogisticRegression(max_iter=1000))
])

In [896]:
# Build the KNN model
# K-Nearest Neighbors (KNN) predicts a class by looking at the nearest data points to a new observation.

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", KNeighborsClassifier())
])


In [897]:
# Build the Decision Tree model
# A Decision Tree makes predictions by asking a series of questions about the data.

tree_pipeline_clf = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", DecisionTreeClassifier(random_state=42))
])

In [898]:
# Compare the three models using Cross-Validation

models_clf = {
    "Logistic Regression": logistic_pipeline,
    "KNN": knn_pipeline,
    "Decision Tree": tree_pipeline_clf
}

clf_results = []

for name, model in models_clf.items():
    scores = cross_validate(
        model,
        X_train_clf,
        y_train_clf,
        cv=cv_clf,
        scoring="f1",
        return_train_score=False
    )


    clf_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

clf_comparison = pd.DataFrame(clf_results)

display(clf_comparison.sort_values(by="Mean F1", ascending=False))

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,KNN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


Mean F1 => tells us the average F1 score across the 5 validation folds.

Std F1 => It tells us how much the F1 scores varied across the 5 folds. A lower standard deviation means the scores were more consistent across those folds.

Logistic Regression: Highest mean F1 (0.392651) among the three models we tested. So it performed best on average across the 5 folds.


In [899]:

# 1. Define the hyperparameter search space
param_grid_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# 2. Create GridSearchCV
grid_search_lr = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid_lr,
    scoring="f1",
    cv=cv_clf,
    n_jobs=-1
)

# 3. Run the hyperparameter search
grid_search_lr.fit(X_train_clf, y_train_clf)

# 4. Display the best parameter and score
print("Best parameters:", grid_search_lr.best_params_)
print("Best CV F1-score:", grid_search_lr.best_score_)

# 5. Display the results for all tested C values
lr_tuning_results = pd.DataFrame(grid_search_lr.cv_results_)

display(
    lr_tuning_results[
        ["param_model__C", "mean_test_score", "std_test_score"]
    ].sort_values(by="mean_test_score", ascending=False)
)


Best parameters: {'model__C': 10}
Best CV F1-score: 0.4225932078257659


,param_model__C,mean_test_score,std_test_score
4,100.00,0.422593,0.088225
3,10.00,0.422593,0.088225
2,1.00,0.392651,0.062747
1,0.10,0.327873,0.046086
0,0.01,0.000000,0.000000


In [900]:
# 1. Get the best tuned model
best_clf_model = grid_search_lr.best_estimator_

# 2. Predict on the test set
y_pred_clf = best_clf_model.predict(X_test_clf)

# 3. Display classification report
print("Classification Report:")
print(classification_report(y_test_clf, y_pred_clf))

# 4. Display confusion matrix
cm_clf = confusion_matrix(y_test_clf, y_pred_clf)

print("Confusion Matrix:")
print(cm_clf)

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180

Confusion Matrix:
[[137   5]
 [ 30   8]]



Logistic Regression was selected for tuning because it had the highest mean F1-score (0.392651) among the three models in the initial comparison. I used GridSearchCV to test five values of C: 0.01, 0.1, 1, 10, and 100, using 5-fold Stratified Cross-Validation. The best C value reported was 10, with a mean CV F1-score of 0.422593. C = 100 achieved the same score.

After tuning, the model was evaluated on the test set. It achieved 81% accuracy. For class 0 (not escalated), the model performed well, with 96% recall. However, for class 1 (escalated), it achieved only 21% recall, meaning it missed many tickets that actually needed escalation.

The confusion matrix showed that the model correctly predicted 137 tickets as not escalated and 8 tickets as escalated. It incorrectly predicted 5 tickets as escalated, while it missed 30 tickets that were actually escalated.

Although the model performed well at identifying tickets that were not escalated, it was less effective at identifying escalated tickets. Since identifying escalated tickets is important, the model's low recall for class 1 is a limitation that should be considered.

## Experiment B: Regression

In [901]:
# Remove ticket_id and the target column from the input features
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])

# Select the target column
y_reg = reg_df["resolution_time_hours"]

# Display the first five rows
display(X_reg.head())

# Display the target values
display(y_reg.head())

,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,7.0,10.0,Medium,Chat,32.638377
1,4.0,15.0,Medium,Chat,45.273760
2,4.0,13.0,Low,Portal,44.878323
3,1.0,10.0,High,Phone,40.095034
4,3.0,15.0,Low,Portal,47.451134


0    8.38724
1    0.50000
2    0.50000
3    0.50000
4    0.50000
Name: resolution_time_hours, dtype: float64

In [902]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train_reg.shape)
print("Testing data shape:", X_test_reg.shape)

Training data shape: (640, 5)
Testing data shape: (160, 5)


In [903]:
# Identify numeric columns
numeric_features_reg = X_train_reg.select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical columns
categorical_features_reg = X_train_reg.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", numeric_features_reg)
print("Categorical features:", categorical_features_reg)

Numeric features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


C:\Users\hp\AppData\Local\Temp\ipykernel_13124\153091838.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_reg = X_train_reg.select_dtypes(


In [904]:
# Numeric preprocessing
numeric_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor_reg = ColumnTransformer(transformers=[
    ("num", numeric_transformer_reg, numeric_features_reg),
    ("cat", categorical_transformer_reg, categorical_features_reg)
])

In [905]:
cv_reg = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [906]:
# Linear Regression pipeline
linear_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("model", LinearRegression())
])

# Ridge Regression pipeline
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

# Decision Tree Regression pipeline
tree_pipeline_reg = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("model", DecisionTreeRegressor(random_state=42))
])

In [907]:
models_reg = {
    "Linear Regression": linear_pipeline,
    "Ridge Regression": ridge_pipeline,
    "Decision Tree Regressor": tree_pipeline_reg
}

print("Three regression models are ready.")

Three regression models are ready.


In [908]:
reg_results = []

for name, model in models_reg.items():

    scores = cross_validate(
        model,
        X_train_reg,
        y_train_reg,
        cv=cv_reg,
        scoring="neg_mean_absolute_error",
        return_train_score=False
    )

    # Convert negative MAE scores into positive values
    mae_scores = -scores["test_score"]

    reg_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

# Create the comparison table
reg_comparison = pd.DataFrame(reg_results)

display(
    reg_comparison.sort_values(
        by="Mean MAE",
        ascending=True
    )
)

,Model,Mean MAE,Std MAE
1,Ridge Regression,2.035127,0.171404
0,Linear Regression,2.035134,0.171650
2,Decision Tree Regressor,3.239097,0.053807


Ridge Regression: Has the lowest MAE, so it has the best average prediction performance.

Linear Regression: Performs almost exactly like Ridge Regression.

Decision Tree Regressor: Has a higher MAE, meaning its predictions have larger average errors.
- We will select Ridge Regression for tuning because it has the lowest Mean MAE.

In [909]:
# Create the parameter grid

param_grid_ridge = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

In [910]:
# Create GridSearchCV

grid_search_ridge = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    scoring="neg_mean_absolute_error",
    cv=cv_reg,
    n_jobs=-1
)

In [911]:
# Train and tune Ridge Regression
grid_search_ridge.fit(X_train_reg, y_train_reg)

print("Best parameters:", grid_search_ridge.best_params_)
print("Best CV MAE:", -grid_search_ridge.best_score_)

Best parameters: {'model__alpha': 0.1}
Best CV MAE: 2.035118085555478


In [912]:
ridge_tuning_results = pd.DataFrame(grid_search_ridge.cv_results_)

display(
    ridge_tuning_results[
        ["param_model__alpha", "mean_test_score", "std_test_score"]
    ].assign(
        mean_MAE=lambda x: -x["mean_test_score"],
        std_MAE=lambda x: x["std_test_score"]
    )[["param_model__alpha", "mean_MAE", "std_MAE"]].sort_values(
        by="mean_MAE",
        ascending=True
    )
)

,param_model__alpha,mean_MAE,std_MAE
1,0.10,2.035118,0.171627
2,1.00,2.035127,0.171404
0,0.01,2.035133,0.171648
3,10.00,2.043030,0.166698
4,100.00,2.209300,0.125366


Alpha = 0.1 has the lowest Mean MAE

Therefore, we select alpha = 0.1

Alpha = 100 performs worse because its Mean MAE is higher

In [913]:
# Get the best Ridge Regression model
best_reg_model = grid_search_ridge.best_estimator_

print("Final model:", best_reg_model)

Final model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['complexity_score',
                                                   'team_experience_years',
                                                   'customer_tenure_months']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
     

In [914]:
# Predict resolution time using the test data
y_pred_reg = best_reg_model.predict(X_test_reg)

# Display the first five predictions
print("First five predictions:", y_pred_reg[:5])

# Display the first five actual values
print("First five actual values:", y_test_reg.iloc[:5].values)

First five predictions: [4.76345948 5.36695324 4.03441313 3.24027424 8.05639023]
First five actual values: [9.1420111  4.88942313 5.25207935 0.66451232 4.21580057]


In [915]:
# Calculate Mean Absolute Error
mae_reg = mean_absolute_error(y_test_reg, y_pred_reg)

# Calculate Root Mean Squared Error
rmse_reg = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print("Final Test MAE:", mae_reg)
print("Final Test RMSE:", rmse_reg)

Final Test MAE: 2.2300861441806563
Final Test RMSE: 3.4027561084563143


### Outliers and MAE vs RMSE

An outlier is a value that is very different from most other values in a dataset.

MAE calculates the average absolute prediction error and treats all errors proportionally.

RMSE gives more importance to larger prediction errors because it squares the errors before calculating the average.

In this experiment, RMSE (3.40) was higher than MAE (2.23). This suggests that some predictions had relatively large errors.

Outliers can affect RMSE more strongly than MAE.


#### 1. Why stratify the classification split?

I use stratification to maintain the same proportion of each class in the training and testing datasets. This is important when the dataset has an imbalanced target variable. It helps ensure that both datasets represent the original class distribution.

#### 2. Why must preprocessing be inside the pipeline?

Preprocessing must be inside the pipeline to prevent data leakage. Each training fold learns preprocessing values, such as the median and scaling parameters, only from its training data. This prevents information from the validation data from affecting the training process.

#### 3. What does fold variability tell you?

Fold variability shows how much a model's performance changes across different cross-validation folds. A low standard deviation means the model performs more consistently, while a high standard deviation means its performance changes more across the folds.

#### 4. Why should the test set be evaluated only once?

The test set should be evaluated only once after selecting and tuning the final model. This provides a fair evaluation of how well the model performs on unseen data. Repeatedly checking the test set could influence model selection and make the evaluation less reliable.

#### 5. What deployment constraints might change your choice?

Deployment constraints such as prediction speed, memory usage, computational cost, interpretability, and maintenance requirements might affect our model choice. For example, a simpler model may be preferred if it is faster, easier to understand, and cheaper to maintain, even if another model has slightly better performance.


## 1. Problem Framing

The purpose of this lab is to build machine learning models for two prediction tasks.

In Experiment A, predict whether a support ticket will be escalated using classification models. The target variable is `escalated`.

In Experiment B, predict how many hours are needed to resolve a support ticket using regression models. The target variable is `resolution_time_hours`.

The goal is to compare different models, select suitable models using cross-validation, tune their hyperparameters, and evaluate their performance on unseen test data.

## 2. Data Preprocessing

Before training the models, I removed the `ticket_id` column because it is an identifier and does not provide useful information for prediction. We separated the target variable from the input features.

I identified numerical and categorical columns and applied different preprocessing techniques.

For numerical columns:
- Missing values were filled using the median.
- StandardScaler was used to scale the numerical features.

For categorical columns:
- Missing values were filled using the most frequent value.
- OneHotEncoder was used to convert categorical values into numerical values.

I used ColumnTransformer to apply the correct preprocessing steps to each type of feature.

## 3. Validation Strategy

For classification, I used an 80% training set and a 20% testing set with stratification. Stratification maintains the proportion of each class in both datasets, which is important because the target classes are imbalanced.

I used 5-fold StratifiedKFold cross-validation to compare the classification models. F1-score was selected as the primary metric because it considers both precision and recall.

For regression, I used an 80% training set and a 20% testing set. I used 5-fold KFold cross-validation and Mean Absolute Error (MAE) to compare the regression models.

## 4. Experiment A: Classification Model Comparison

I compared three classification models: Logistic Regression, K-Nearest Neighbors (KNN), and Decision Tree Classifier.

The initial cross-validation results were:

| Model | Mean F1 | Std F1 |
|---|---:|---:|
| Logistic Regression | 0.392651 | 0.062747 |
| KNN | 0.360835 | 0.050295 |
| Decision Tree | 0.330666 | 0.058126 |

Logistic Regression achieved the highest mean F1-score, so we selected it for hyperparameter tuning.

### Hyperparameter Tuning

I used GridSearchCV to test different values of C:

`[0.01, 0.1, 1, 10, 100]`

The best reported parameter was C = 10, with a mean cross-validation F1-score of 0.422593. C = 100 achieved the same score.

### Final Classification Test Results

After tuning, I evaluated the selected Logistic Regression model on the test set.

- Accuracy: 81%
- Class 0 recall: 96%
- Class 1 precision: 62%
- Class 1 recall: 21%
- Class 1 F1-score: 0.31

The confusion matrix was:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 137 | 5 |
| Actual 1 | 30 | 8 |

The model correctly predicted 137 tickets as not escalated and 8 tickets as escalated. It produced 5 false positives and 30 false negatives.

This shows that the model identifies tickets that are not escalated well, but it misses many tickets that actually need escalation. Therefore, its low recall for class 1 is an important limitation.

## 5. Experiment B: Regression Model Comparison

I compared Linear Regression, Ridge Regression, and Decision Tree Regressor using 5-fold cross-validation and MAE.

The results were:

| Model | Mean MAE | Std MAE |
|---|---:|---:|
| Ridge Regression | 2.035127 | 0.171404 |
| Linear Regression | 2.035134 | 0.171650 |
| Decision Tree Regressor | 3.239097 | 0.053807 |

Ridge Regression achieved the lowest mean MAE, although its performance was almost identical to Linear Regression. Therefore, we selected Ridge Regression for tuning.

### Hyperparameter Tuning

We used GridSearchCV to test five alpha values:

`[0.01, 0.1, 1, 10, 100]`

The best alpha value was 0.1, with a cross-validation MAE of approximately 2.035118.

### Final Regression Test Results

After tuning, we evaluated Ridge Regression on the test set.

- MAE: 2.23 hours
- RMSE: 3.40 hours

The MAE means that the model's predictions differed from the actual resolution times by an average of approximately 2.23 hours.

RMSE was higher than MAE, suggesting that some predictions had relatively large errors.

## 6. Outliers and MAE vs RMSE

An outlier is a value that is very different from most other values in a dataset.

MAE calculates the average absolute prediction error and treats errors proportionally. RMSE gives more importance to larger errors because it squares the errors before calculating the average.

Therefore, MAE and RMSE may prefer different models. A model with smaller average errors may still have some large errors that increase its RMSE.

## 7. Leakage Prevention

I placed preprocessing steps inside the machine learning pipelines. This ensures that imputation, scaling, and encoding are fitted using only the training portion of each cross-validation fold.

The validation data is only transformed using the preprocessing values learned from the training data. This prevents information from the validation data from leaking into model training.

I also kept the test sets separate during model comparison and hyperparameter tuning. The selected models were evaluated on the test sets only after the selection and tuning processes.

## 8. Final Model Justification and Limitations

For classification, Logistic Regression was selected because it achieved the highest mean F1-score among the three initial models. However, its recall for escalated tickets was only 21%, meaning it missed many tickets that actually needed escalation. Further improvement is needed before using it in a real support-ticket system.

For regression, Ridge Regression was selected because it achieved the lowest mean MAE. Its test MAE was 2.23 hours. However, the model can still make relatively large prediction errors, as indicated by its RMSE of 3.40 hours.

The model choices could also change depending on deployment requirements, such as prediction speed, computational resources, interpretability, and maintenance costs.
